In [ ]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader

from medicalDataLoader_Enes import MedicalImageDataset
from utils_Enes import (device, DiceLoss, FocalLoss, evaluate, predict, keep_largest_component, dice_per_class,
                        hausdorff_per_class, asd_per_class, show_predictions, show_confusion)
from UNet_Base_Enes import UNet

In [ ]:
EPOCHS = 100
BATCH_SIZE = 8
LR = 0.001
SEED = 0
DOUBLE_CONV = False
WEIGHTED_CE = False
FOCAL = False
MODEL_DIR = 'models/final_dice'
if DOUBLE_CONV:
    MODEL_DIR += '_double'
if WEIGHTED_CE:
    MODEL_DIR += '_weighted'
if FOCAL:
    MODEL_DIR += '_focal'

if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/datasets/enesozkosar/acdc-heartbeat/Data/'
else:
    DATA_DIR = '../../Data/'

In [ ]:
random.seed(SEED)
torch.manual_seed(SEED)

train_set = MedicalImageDataset('train', DATA_DIR, augment=True)
val_set = MedicalImageDataset('val', DATA_DIR)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_set, batch_size=BATCH_SIZE)

net = UNet(num_classes=4, double=DOUBLE_CONV).to(device)

ce_weights = None
if WEIGHTED_CE:
    counts = torch.zeros(4)
    for _, target in MedicalImageDataset('train', DATA_DIR):
        counts += torch.bincount(target.flatten(), minlength=4)
    freq = counts / counts.sum()
    ce_weights = (freq.median() / freq).sqrt().to(device)
    print('Poids des classes (fond, VD, Myo, VG) :', ce_weights.cpu().numpy().round(3))
if FOCAL:
    ce_loss = FocalLoss(gamma=2)
else:
    ce_loss = nn.CrossEntropyLoss(weight=ce_weights)
dice_loss = DiceLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

In [ ]:
os.makedirs(MODEL_DIR, exist_ok=True)
history = {'train_loss': [], 'val_dice': [], 'val_hd': []}
best_dice = -1

for epoch in range(EPOCHS):
    net.train()
    losses = []
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        out = net(images)
        loss = ce_loss(out, targets) + dice_loss(out, targets)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    scheduler.step()

    dice, hd = evaluate(net, val_loader)
    history['train_loss'].append(np.mean(losses))
    history['val_dice'].append(dice)
    history['val_hd'].append(hd)
    print('Epoch {} | loss {:.4f} | Dice VD {:.3f} Myo {:.3f} VG {:.3f} | HD VD {:.1f} Myo {:.1f} VG {:.1f}'.format(
        epoch, np.mean(losses), *dice, *hd))

    if np.mean(dice) > best_dice:
        best_dice = np.mean(dice)
        torch.save(net.state_dict(), MODEL_DIR + '/best.pth')

torch.save(net.state_dict(), MODEL_DIR + '/last.pth')
np.save(MODEL_DIR + '/history.npy', history)

In [ ]:
plt.plot(np.mean(history['val_dice'], axis=1))
plt.xlabel('Époque')
plt.ylabel('Dice moyen (validation)')
plt.show()

net.load_state_dict(torch.load(MODEL_DIR + '/last.pth', map_location=device))
pred, target = predict(net, val_loader)
pred = keep_largest_component(pred)
dice = dice_per_class(pred, target)
hd = hausdorff_per_class(pred, target)
asd = asd_per_class(pred, target)
print(MODEL_DIR, '| paramètres : {:,}'.format(sum(p.numel() for p in net.parameters())))
print('Dice moyen des 10 dernières époques : {:.3f}'.format(np.mean(history['val_dice'][-10:])))
print('Dernier modèle | Dice VD {:.3f} Myo {:.3f} VG {:.3f} | HD VD {:.1f} Myo {:.1f} VG {:.1f} | ASD VD {:.2f} Myo {:.2f} VG {:.2f}'.format(
    *dice, *hd, *asd))

show_predictions(net, val_set, [5, 20, 40, 60], post_process=True)
show_confusion(net, val_loader, post_process=True)